# Candle Prediction using Market Depth

In [129]:
from pathlib import Path
import pandas as pd
import numpy as np
from utils import resample_fractional_minute, apply_trailing_logic

In [130]:
# ---- Input ------
date_ = "23APR2026"
file_name = "NIFTY26APR24200CE.xlsx"

file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)
if "PE" in file_name or "CE" in file_name:
    col_name = "last_trade_time"

    # Volume computation
    # 1. Calculate the basic difference between rows
    df["volume_at_tick"] = df["volume_traded"].diff()
    df.loc[df["volume_at_tick"] == 0, "volume_at_tick"] = np.nan
    df["volume_at_tick"] = df["volume_at_tick"].ffill()
    df["volume_at_tick"] = df["volume_at_tick"].fillna(0)
else:
    col_name = "local_time"

df[col_name] = pd.to_datetime(df[col_name])
target_date = pd.to_datetime(date_).date()
df = df[df[col_name].dt.date == target_date]

In [131]:
file_name

'NIFTY26APR24200CE.xlsx'

In [132]:
df.head(4)

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,total_sell_quantity,ohlc,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_at_tick
2,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
3,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
4,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
5,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0


In [133]:
# ---- Input ------
N = 6
window = 3
volume_period = 30

In [134]:
df.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick'],
      dtype='str')

In [135]:
print(type(df.iloc[0]["local_time"]))
print(df.iloc[0]["local_time"])
print(df.iloc[0]["last_trade_time"])

<class 'pandas.Timestamp'>
2026-04-23 09:15:00.619000
2026-04-23 09:15:00


In [136]:
clubbed_df = resample_fractional_minute(df, col_name, N)
clubbed_df["bucket_time_next"] = clubbed_df["bucket_time"].shift(-1)
clubbed_df["price_diff"] = clubbed_df["close"] - clubbed_df["open"]
clubbed_df["price_pct"] = (clubbed_df["close"] - clubbed_df["open"])/clubbed_df["open"]

clubbed_df["volume_diff"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_participated"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_pct"] = (clubbed_df["volume_close"] - clubbed_df["volume_open"])/clubbed_df["volume_open"]
clubbed_df["volume_avg"] = clubbed_df["volume_close"].rolling(volume_period).median()

In [137]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].head()

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,minute_high,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg
270,2026-04-23 10:00:00,2026-04-23 10:00:00,243.20,243.65,241.00,243.65,10010.0,10010.0,2015.0,8840.0,...,243.65,235.65,235.8,2026-04-23 10:00:10,0.45,0.001850,7995.0,7995.0,-0.116883,5817.5
271,2026-04-23 10:00:10,2026-04-23 10:00:00,242.30,243.60,241.25,241.25,5070.0,5525.0,1820.0,2275.0,...,243.65,235.65,235.8,2026-04-23 10:00:20,-1.05,-0.004333,3705.0,3705.0,-0.551282,5590.0
272,2026-04-23 10:00:20,2026-04-23 10:00:00,241.95,242.90,241.10,241.10,3965.0,6630.0,1430.0,6630.0,...,243.65,235.65,235.8,2026-04-23 10:00:30,-0.85,-0.003513,5200.0,5200.0,0.672131,5590.0
273,2026-04-23 10:00:30,2026-04-23 10:00:00,241.55,241.80,238.60,240.20,1690.0,9620.0,1235.0,4680.0,...,243.65,235.65,235.8,2026-04-23 10:00:40,-1.35,-0.005589,8385.0,8385.0,1.769231,5102.5
274,2026-04-23 10:00:40,2026-04-23 10:00:00,239.65,239.65,236.10,236.10,2665.0,39195.0,2275.0,6305.0,...,243.65,235.65,235.8,2026-04-23 10:00:50,-3.55,-0.014813,36920.0,36920.0,1.365854,5102.5


In [138]:
clubbed_df[["volume_open", "volume_high", "volume_low", "volume_close", "volume_participated"]]

,volume_open,volume_high,volume_low,volume_close,volume_participated
0,15665.0,118235.0,15665.0,74360.0,102570.0
1,68380.0,81900.0,40625.0,40625.0,41275.0
2,46735.0,46735.0,30810.0,30810.0,15925.0
3,40690.0,89635.0,20995.0,33410.0,68640.0
4,51935.0,53885.0,25935.0,26260.0,27950.0
...,...,...,...,...,...
2245,24505.0,27755.0,12285.0,12285.0,15470.0
2246,19565.0,24960.0,15405.0,18980.0,9555.0
2247,25870.0,25870.0,7800.0,14690.0,18070.0
2248,17485.0,17485.0,7215.0,11115.0,10270.0


In [139]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].shape

(36, 22)

In [140]:
def generate_signal(
    df,
    window: int,
    price_pct_threshold = 0.001,
    volume_threshold = 0,

    use_price_pct_level=False,
    use_price_trend=True,
    use_volume=False
):

    print("use_price_pct_level : ", use_price_pct_level)
    print("use_price_trend : ", use_price_trend)
    print("use_volume : ", use_volume)
    # ---------------------------
    # BASE CONDITIONS (level)
    # ---------------------------


    if use_price_pct_level:
        cond_buy = df["price_pct"] > price_pct_threshold
        cond_sell = df["price_pct"] < -price_pct_threshold
        buy_threshold_streak = cond_buy.rolling(window).min() == 1
        sell_threshold_streak = cond_sell.rolling(window).min() == 1
    else:
        buy_threshold_streak = pd.Series(True, index=df.index)
        sell_threshold_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # PRICE TREND (monotonic)
    # ---------------------------
    if use_price_trend:
        price_diff = df["close"].diff()

        cond_buy_trend = price_diff > 0
        cond_sell_trend = price_diff < 0

        buy_trend_streak = cond_buy_trend.rolling(window).min() == 1
        sell_trend_streak = cond_sell_trend.rolling(window).min() == 1
    else:
        buy_trend_streak = pd.Series(True, index=df.index)
        sell_trend_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # VOLUME CONDITION
    # ---------------------------
    if use_volume:
        # print("use_volume : ", use_volume)
        # vol_diff = df["volume_participated"].diff()
        # cond_vol = vol_diff >= 0
        # vol_streak = cond_vol.rolling(window).min() == 1

        cond_vol = df["volume_close"] > (df["volume_avg"] * 1.2)
        vol_streak = cond_vol.rolling(window).min() == 1
    else:
        vol_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # FINAL STREAKS
    # ---------------------------
    buy_streak = buy_threshold_streak & buy_trend_streak & vol_streak
    sell_streak = sell_threshold_streak & sell_trend_streak & vol_streak

    # ---------------------------
    # SIGNAL
    # ---------------------------
    # df["predicted"] = np.where(
    #     buy_streak, "BUY",
    #     np.where(sell_streak, "SELL", None)
    # )
    conditions = [
        buy_streak & ~sell_streak,
        sell_streak & ~buy_streak
    ]
    choices = ["BUY", "SELL"]
    df["predicted"] = np.select(conditions, choices, default=None)
    return df

In [141]:
# generate_signal
clubbed_df_2 = generate_signal(clubbed_df, window, price_pct_threshold = 0.002, volume_threshold=1000,
                               use_price_pct_level=True, use_price_trend=True, use_volume=True)

use_price_pct_level :  True
use_price_trend :  True
use_volume :  True


In [142]:
clubbed_df_2[clubbed_df_2["predicted"]==clubbed_df["candle_type"]].tail(5)

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg,predicted
1477,2026-04-23 13:21:10,2026-04-23 13:21:00,210.20,210.65,208.90,209.75,28145.0,48945.0,23010.0,23010.0,...,208.90,211.0,2026-04-23 13:21:20,-0.45,-0.002141,25935.0,25935.0,-0.182448,16510.0,SELL
1571,2026-04-23 13:36:50,2026-04-23 13:36:00,202.40,202.65,201.00,201.00,6825.0,9490.0,3705.0,9230.0,...,201.00,201.0,2026-04-23 13:37:00,-1.40,-0.006917,5785.0,5785.0,0.352381,5817.5,SELL
1738,2026-04-23 14:04:40,2026-04-23 14:04:00,196.15,196.35,194.55,194.85,2665.0,12415.0,2535.0,12415.0,...,194.55,196.0,2026-04-23 14:04:50,-1.30,-0.006628,9880.0,9880.0,3.658537,6045.0,SELL
1849,2026-04-23 14:23:10,2026-04-23 14:23:00,182.10,183.65,182.10,183.00,16640.0,16640.0,8710.0,13650.0,...,181.50,183.2,2026-04-23 14:23:20,0.90,0.004942,7930.0,7930.0,-0.179688,10140.0,BUY
2146,2026-04-23 15:12:40,2026-04-23 15:12:00,163.05,163.15,161.10,161.85,11635.0,28795.0,11635.0,23205.0,...,161.10,163.3,2026-04-23 15:12:50,-1.20,-0.007360,17160.0,17160.0,0.994413,5622.5,SELL


In [143]:
# df_with_signal = df.merge(
#         clubbed_df[["bucket_time", "predicted"]],
#         left_on="last_trade_time",
#         right_on="bucket_time",
#         how="left"
#     )

# import pandas as pd

# 1. Ensure both DataFrames are sorted by the time columns
df = df.sort_values("last_trade_time")
clubbed_df_2 = clubbed_df_2.sort_values("bucket_time")

# 2. Perform the proximity merge
df_with_signal = pd.merge_asof(
    df,
    clubbed_df_2[["bucket_time", "predicted"]],
    left_on="last_trade_time",
    right_on="bucket_time",
    direction="backward" # Only looks at the past/current, never the future
)

# Find duplicates in bucket_time and set their 'predicted' value to NaN
df_with_signal.loc[df_with_signal.duplicated(subset=['bucket_time'], keep='first'), 'predicted'] = np.nan

In [144]:
# df_with_signal.to_excel("df_with_signal.xlsx")

In [145]:
# clubbed_df_2.to_excel("clubbed_df_2.xlsx")

In [146]:
df_with_signal.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_at_tick,bucket_time,predicted
0,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
1,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
2,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
3,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
4,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:02.120,2026-04-23 09:15:01,184.05,130,191.93,CE,atm,...,-43.818681,1252485,1252485,1252485,"{'buy': [{'quantity': 910, 'price': 182.55, 'o...",True,full,105430.0,2026-04-23 09:15:00,NaN


In [147]:
df_with_signal.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick', 'bucket_time', 'predicted'],
      dtype='str')

In [148]:
params = {
    "initial_sl_pct": 0.02,
    "target_pct": 0.01,
    "trail_sl_pct": 0.02,
    "tight_sl_offset": 0.5,
}

In [149]:
trades = apply_trailing_logic(df_with_signal, params)

trades = pd.DataFrame(trades)
if len(trades):
    trades["final"] = trades.apply(lambda row: "profit" if row["profit"] > 0 else "loss", axis=1)
else:
    print("trades not generated")

In [150]:
len(trades)

16

In [151]:
# trades

In [152]:
trades[trades["final"]=="profit"]["profit"].sum()

np.float64(29.899999999999977)

In [153]:
trades[trades["final"]=="loss"]["profit"].sum()

np.float64(-5.251000000000005)

In [154]:
trades.head(11)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final
0,2026-04-23 09:21:30,200.85,2026-04-23 09:21:45,202.40,1.55,0.007717,profit
1,2026-04-23 09:31:10,215.55,2026-04-23 09:31:16,218.40,2.85,0.013222,profit
2,2026-04-23 09:39:10,204.20,2026-04-23 09:39:16,206.35,2.15,0.010529,profit
3,2026-04-23 10:03:40,248.40,2026-04-23 10:03:47,250.70,2.30,0.009259,profit
4,2026-04-23 10:04:20,253.35,2026-04-23 10:05:25,256.45,3.10,0.012236,profit
5,2026-04-23 10:16:31,230.35,2026-04-23 10:16:34,233.20,2.85,0.012372,profit
6,2026-04-23 10:47:40,185.55,2026-04-23 10:47:43,187.70,2.15,0.011587,profit
7,2026-04-23 10:58:40,192.90,2026-04-23 10:58:52,194.35,1.45,0.007517,profit
8,2026-04-23 11:45:20,176.15,2026-04-23 11:45:29,178.45,2.30,0.013057,profit
9,2026-04-23 12:07:30,185.75,2026-04-23 12:09:00,188.25,2.50,0.013459,profit


In [155]:
trades["final"].value_counts()

final
profit    14
loss       2
Name: count, dtype: int64

In [156]:
trades["final"].value_counts(normalize=True) * 100

final
profit    87.5
loss      12.5
Name: proportion, dtype: float64

In [157]:
trades[trades["final"]=="loss"].head(12)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final
13,2026-04-23 13:14:40,186.65,2026-04-23 13:15:57,184.142,-2.508,-0.013437,loss
14,2026-04-23 14:04:11,198.40,2026-04-23 14:04:42,195.657,-2.743,-0.013826,loss


In [160]:
trades[trades["final"]=="loss"].columns

Index(['entry_time', 'entry_price', 'exit_time', 'exit_price', 'profit',
       'profit_pct', 'final'],
      dtype='str')

In [158]:

# clubbed_df_2.to_excel(Path(f"assets/logs/{date_}/clubbed_df.xlsx"))

In [159]:
# clubbed_df_2["predicted"].value_counts()